# This notebook contains a handwritten implementation of the core business logic

## Setup and Installation



In [ ]:
!pip install -q langchain-openai langchain-chroma langchain-huggingface \
    langchain-community langchain-text-splitters pypdf gradio

## Library Imports

Import all necessary modules: `ChatOpenAI`, `Chroma`, `HuggingFaceEmbeddings`, `PyPDFLoader`, `RecursiveCharacterTextSplitter`, and utilities for type hinting (`Enum`, `Path`, `List`, `Optional`, `BaseModel`, `Field`, `model_validator`, `defaultdict`), along with `display` and `Markdown`.

In [3]:
from huggingface_hub import login
from google.colab import userdata
from langchain_openai import ChatOpenAI
from langchain_chroma import Chroma
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import CharacterTextSplitter, RecursiveCharacterTextSplitter
from langchain_core.retrievers import BaseRetriever
from enum import Enum
from pathlib import Path
from typing import List, Optional
from pydantic import BaseModel, Field, model_validator
from collections import defaultdict
from IPython.display import display, Markdown

/tmp/ipykernel_5474/3150001891.py:7: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


## API Key Setup and LLM Initialization

Retrieve API keys for Hugging Face and OpenRouter from Colab secrets. Initialize `ChatOpenAI` with `openai/gpt-4o-mini` via OpenRouter. Declare global variables for the vector store and retriever, to be lazily initialized.

In [13]:
hf_token = userdata.get("HF_TOKEN")
login(hf_token, add_to_git_credential=True)

open_router = userdata.get("OPENROUTER_API_KEY")

llm = ChatOpenAI(
    model="openai/gpt-4o-mini",
    api_key=open_router,
    base_url="https://openrouter.ai/api/v1",
)

# RAG state
_vectorstore: Optional[Chroma] = None
_retriever: Optional[BaseRetriever] = None

## System Prompt Template

Defines `SYSTEM_PROMPT_TEMPLATE`, which guides the AI assistant to answer questions strictly based on provided context and conversation history, adhering to specific rules.

In [6]:
SYSTEM_PROMPT_TEMPLATE = """
You are a document-grounded AI assistant. Your sole source of truth is the
context provided below. Answer the user's question strictly based on that context.

## Rules

- Answer ONLY based on the provided context.
- If the context does not contain enough information to answer, say so explicitly.
- Never speculate or use outside knowledge to fill gaps.
- Be concise and direct. Lead with the answer, then support it with evidence from the context.
- If multiple parts of the context contain conflicting information, surface the
  conflict explicitly rather than picking one silently.
- Do not reveal the raw context text verbatim unless the user explicitly asks for it.

## Conversation history
{history}

## Context
{context}
"""

## Data Models and Document Processing Utilities

Defines `InputType` and `UserInput` for structuring input (prompt, PDF, or text). The `chunking` function splits documents into smaller chunks using `RecursiveCharacterTextSplitter`. `build_vectorstore` creates a `Chroma` vector store with `HuggingFaceEmbeddings`, and `make_retriever` configures a retriever from this store.

In [21]:
class InputType(str, Enum):
    PDF = "pdf"
    TEXT = "text"


class UserInput(BaseModel):
    user_prompt: str = Field(description="The user prompt")
    doc_input_type: Optional[InputType] = Field(
        default=None, description="Type of document input: pdf or text"
    )
    raw_text: Optional[str] = None
    pdf_files: Optional[List[Path]] = None

    @model_validator(mode="after")
    def validate_content(self):
        if self.doc_input_type == InputType.TEXT:
            if not self.raw_text:
                raise ValueError("raw_text must be provided when doc_input_type='text'")
            self.pdf_files = None
        elif self.doc_input_type == InputType.PDF:
            if not self.pdf_files:
                raise ValueError("pdf_files must be provided when doc_input_type='pdf'")
            self.raw_text = None
        return self


def chunking(document_input: UserInput) -> list:
    """Split a document into LangChain Document chunks."""
    # Use RecursiveCharacterTextSplitter for ALL input types
    # Larger chunk_size keeps table rows intact
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=800,       # was 400 — too small for tables
        chunk_overlap=100,    # was 50 — more overlap bridges split context
        separators=["\n\n", "\n", " ", ""],  # respects paragraph > line > word
    )

    chunks = []
    if document_input.doc_input_type == InputType.TEXT:
        chunks = splitter.create_documents([document_input.raw_text])

    elif document_input.doc_input_type == InputType.PDF:
        for pdf_path in document_input.pdf_files:
            if not pdf_path.exists():
                raise FileNotFoundError(f"PDF not found: {pdf_path}")
            docs = PyPDFLoader(str(pdf_path)).load()
            chunks.extend(splitter.split_documents(docs))

    return chunks


def build_vectorstore(chunks) -> Chroma:
    """Create a new Chroma vectorstore from chunks."""
    embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
    return Chroma.from_documents(chunks, embeddings)


def make_retriever(vectorstore: Chroma, k: int = 4, search_type: str = "similarity"):
    return vectorstore.as_retriever(search_type=search_type, search_kwargs={"k": k})

## RAG Pipeline Core Logic

-   **`update_vectorstore`**: Lazily creates or updates the global `_vectorstore` and `_retriever` by processing new documents into chunks and adding them to the vector store. Refreshes the retriever.
-   **`answer_question`**: Runs the RAG pipeline. Updates the vector store, retrieves relevant `docs` based on the prompt, formats them into `context`, and uses the `SYSTEM_PROMPT_TEMPLATE` and `history` to generate an answer via `llm`.

In [22]:
def update_vectorstore(user_input: UserInput, k: int = 10) -> None:
    """
    Lazily create or incrementally update the global vectorstore.
    Only processes documents when new content is provided.
    """
    global _vectorstore, _retriever

    has_content = user_input.pdf_files or user_input.raw_text
    if not has_content:
        return  # question-only turn; reuse existing retriever

    chunks = chunking(user_input)
    if not chunks:
        return

    if _vectorstore is None:
        _vectorstore = build_vectorstore(chunks)
    else:
        _vectorstore.add_documents(chunks)

    # Always refresh the retriever after any change to the vectorstore
    _retriever = make_retriever(_vectorstore, k=k)


def answer_question(user_input: UserInput, history: str, k: int = 6):
    """
    Run the full RAG pipeline.

    Returns
    -------
    answer : str
    docs   : list[Document]
    """
    global _retriever

    update_vectorstore(user_input, k=k)

    if _retriever is None:
        return (
            "No document has been loaded yet. Please upload a file or provide text first.",
            [],
        )

    docs = _retriever.invoke(user_input.user_prompt)
    context = "\n\n".join(doc.page_content for doc in docs)
    system_prompt = SYSTEM_PROMPT_TEMPLATE.format(context=context, history=history)

    response = llm.invoke(
        [SystemMessage(content=system_prompt), HumanMessage(content=user_input.user_prompt)]
    )
    return response.content, context

## Document Display Utility

`display_docs` pretty-prints retrieved document chunks in Markdown format within notebooks, grouping by source and showing metadata and content for easy review.

In [23]:
def display_docs(docs) -> None:
    """Pretty-print retrieved chunks as Markdown in Jupyter / Colab."""
    grouped: dict = defaultdict(list)
    for i, doc in enumerate(docs, start=1):
        source = doc.metadata.get("source", "Unknown Source")
        grouped[source].append((i, doc))

    md = "# Retrieved Documents\n\n"
    for source, chunks in grouped.items():
        md += f"## 📄 `{source}`\n\n"
        md += f"Retrieved chunks: **{len(chunks)}**\n\n"
        for chunk_num, doc in chunks:
            md += f"### Chunk #{chunk_num}\n\n"
            if doc.metadata:
                md += "**Metadata**\n\n"
                for key, val in doc.metadata.items():
                    md += f"- **{key}**: {val}\n"
                md += "\n"
            md += "**Content**\n\n```text\n"
            md += doc.page_content.strip()
            md += "\n```\n\n"

    display(Markdown(md))

## Gradio User Interface

Sets up the Gradio UI for the RAG chatbot:

-   `_conversation_history`: Stores chat history.
-   `format_context`: Formats retrieved context for UI display.
-   `_format_history`: Formats history for LLM prompt.
-   `chat`: Processes user input (files/text), calls `answer_question`, updates history, and formats context.
-   `clear_all`: Resets chat interface and history.
-   `gr.Blocks`: Defines UI layout (chatbot, input, file upload, buttons).
-   Event Handlers: Links UI actions to `chat` and `clear_all`.

`demo.launch()` starts the Gradio interface.

In [28]:
import gradio as gr
from pathlib import Path

_conversation_history: list[tuple[str, str]] = []

def format_context(context):
    result = "<h2 style='color: #ff7800;'>Relevant Context</h2>\n\n"
    for doc in context:
        result += f"<span style='color: #ff7800;'>Source: {doc.metadata['source']}</span>\n\n"
        result += f"<div style='background: #1f2937; padding: 12px; border-radius: 6px; margin: 8px 0;'>"
        result += doc.page_content
        result += "</div>\n\n"
    return result if context else "<p style='color: #888;'>No context retrieved yet.</p>"


def _format_history(pairs: list[tuple[str, str]]) -> str:
    if not pairs:
        return "No prior conversation."
    return "\n".join(f"{role.upper()}: {text}" for role, text in pairs)


def chat(user_message: str, uploaded_files, chat_history: list):
    global _conversation_history

    if not user_message.strip():
        return "", None, chat_history, ""

    # ── Build UserInput ──────────────────────────────────────────────
    if uploaded_files:
        pdf_paths = [Path(f) for f in uploaded_files if f.lower().endswith(".pdf")]
        if pdf_paths:
            user_input = UserInput(
                user_prompt=user_message,
                doc_input_type=InputType.PDF,
                pdf_files=pdf_paths,
            )
        else:
            raw = ""
            for f in uploaded_files:
                try:
                    raw += Path(f).read_text(errors="ignore")[:8_000]
                except Exception:
                    pass
            user_input = UserInput(
                user_prompt=user_message,
                doc_input_type=InputType.TEXT,
                raw_text=raw or "No readable content found.",
            )
    else:
        user_input = UserInput(user_prompt=user_message, doc_input_type=None)

    # ── RAG ──────────────────────────────────────────────────────────
    try:
        answer, context = answer_question(
            user_input, history=_format_history(_conversation_history)
        )
    except Exception as exc:
        answer = f"⚠️ Error: {exc}"
        context = []

    # ── Update histories ─────────────────────────────────────────────
    _conversation_history.extend([("user", user_message), ("assistant", answer)])
    chat_history.extend(
        [
            {"role": "user", "content": user_message},
            {"role": "assistant", "content": answer},
        ]
    )

    return "", None, chat_history, format_context(context)


def clear_all():
    global _conversation_history
    _conversation_history = []
    return [], "", None, ""


# ── Layout ───────────────────────────────────────────────────────────
with gr.Blocks(title="RAG Chat", theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 📚 Document-Grounded RAG Chat")
    gr.Markdown(
        "Upload a **PDF** or **.txt** file to ground the conversation, "
        "then ask questions."
    )

    with gr.Row(equal_height=True):
        # Left Column - Chat
        with gr.Column(scale=3):
            chatbot = gr.Chatbot(
                height=520,
                label="Conversation",
                type="messages",
                show_copy_button=True
            )

            with gr.Row():
                msg_box = gr.Textbox(
                    placeholder="Ask a question…",
                    show_label=False,
                    scale=4,
                    container=False
                )
                send_btn = gr.Button("Send", variant="primary", scale=1)

            file_upload = gr.File(
                label="📎 Upload document (PDF or .txt)",
                file_count="multiple",
                file_types=[".pdf", ".txt"],
                type="filepath",
            )

            clear_btn = gr.Button(
                "🗑️ Clear conversation & reset", variant="secondary"
            )

        # Right Column - Context Panel
        with gr.Column(scale=2):
            context_display = gr.HTML(
                label="📖 Relevant Context",
                value="<p style='color: #888;'>Context will appear here after your first question.</p>",
                show_label=True,
                elem_id="context-panel"
            )

    # ── Event Handlers ───────────────────────────────────────────────
    send_btn.click(
        chat,
        inputs=[msg_box, file_upload, chatbot],
        outputs=[msg_box, file_upload, chatbot, context_display]
    )

    msg_box.submit(
        chat,
        inputs=[msg_box, file_upload, chatbot],
        outputs=[msg_box, file_upload, chatbot, context_display]
    )

    clear_btn.click(
        clear_all,
        outputs=[chatbot, msg_box, file_upload, context_display]
    )

demo.launch(debug=False)

/tmp/ipykernel_5474/1437731449.py:80: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(title="RAG Chat", theme=gr.themes.Soft()) as demo:
/tmp/ipykernel_5474/1437731449.py:90: DeprecationWarning: The 'show_copy_button' parameter will be removed in Gradio 6.0. You will need to use 'buttons=["copy"]' instead.
  chatbot = gr.Chatbot(
/tmp/ipykernel_5474/1437731449.py:90: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot = gr.Chatbot(
/usr/local/lib/python3.12/dist-packages/gradio/components/base.py:200: UserWarning: show_label has no effect when container is False.
  warnings.warn("show_label has no effect when container is False.")


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://186c98002557da93e8.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
